# Machine Learning 1 - Nearest Neighbors and Decision Trees

## Lab objectives

* Classification with decision trees and random forests.
* Cross-validation and evaluation.

In [ ]:
from lab_tools import CIFAR10, get_hog_image

dataset = CIFAR10('./CIFAR10')

Pre-loading training data
Pre-loading test data


# 1. Nearest Neighbor

The following example uses the Nearest Neighbor algorithm on the Histogram of Gradient decriptors in the dataset.

In [3]:
from sklearn.neighbors import KNeighborsClassifier

clf = KNeighborsClassifier(n_neighbors=1)
clf.fit( dataset.train['hog'], dataset.train['labels'] )

KNeighborsClassifier(n_neighbors=1)

* What is the **descriptive performance** of this classifier ?
* Modify the code to estimate the **predictive performance**.
* Use cross-validation to find the best hyper-parameters for this method.

In [4]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold, cross_val_score
import numpy as np

X_train = dataset.train['hog']
y_train = dataset.train['labels']
X_test = dataset.test['hog']
y_test = dataset.test['labels']

# 1. Descriptive performance: performance on training set
clf = KNeighborsClassifier(n_neighbors=1)
clf.fit(X_train, y_train)

train_pred = clf.predict(X_train)
train_acc = accuracy_score(y_train, train_pred)
print("Descriptive performance (train accuracy):", train_acc)
print("Training confusion matrix:")
print(confusion_matrix(y_train, train_pred))

# 2. Predictive performance: performance on test set
test_pred = clf.predict(X_test)
test_acc = accuracy_score(y_test, test_pred)
print("\nPredictive performance (test accuracy):", test_acc)
print("Test confusion matrix:")
print(confusion_matrix(y_test, test_pred))

# 3. Cross-validation to find the best hyper-parameter
k_values = [1, 3, 5, 7, 9]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

best_k = None
best_score = 0

print("\nCross-validation results:")
for k in k_values:
    clf_cv = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(clf_cv, X_train, y_train, cv=cv, scoring="accuracy")
    mean_score = scores.mean()
    print(f"k = {k} -> mean CV accuracy = {mean_score:.4f}")
    
    if mean_score > best_score:
        best_score = mean_score
        best_k = k

print("\nBest k:", best_k)
print("Best CV accuracy:", best_score)


Descriptive performance (train accuracy): 1.0
Training confusion matrix:
[[5000    0    0]
 [   0 5000    0]
 [   0    0 5000]]

Predictive performance (test accuracy): 0.694
Test confusion matrix:
[[609 258 133]
 [ 63 754 183]
 [ 26 255 719]]

Cross-validation results:
k = 1 -> mean CV accuracy = 0.6885
k = 3 -> mean CV accuracy = 0.7078
k = 5 -> mean CV accuracy = 0.7111
k = 7 -> mean CV accuracy = 0.7073
k = 9 -> mean CV accuracy = 0.7060

Best k: 5
Best CV accuracy: 0.7111333333333334


## 2. Decision Trees

[Decision Trees](http://scikit-learn.org/stable/modules/tree.html#tree) classify the data by splitting the feature space according to simple, single-feature rules. Scikit-learn uses the [CART](https://en.wikipedia.org/wiki/Predictive_analytics#Classification_and_regression_trees_.28CART.29) algorithm for [its implementation](http://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html) of the classifier. 

* **Create a simple Decision Tree classifier** using scikit-learn and train it on the HoG training set.
* Use cross-validation to find the best hyper-paramters for this method.

In [5]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, confusion_matrix
from lab_tools import CIFAR10

if 'dataset' not in globals():
    dataset = CIFAR10('./CIFAR10')

X_train = dataset.train['hog']
y_train = dataset.train['labels']
X_test = dataset.test['hog']
y_test = dataset.test['labels']

# Simple Decision Tree
clf = DecisionTreeClassifier(random_state=42)
clf.fit(X_train, y_train)

# Test performance
y_pred = clf.predict(X_test)
print("Test accuracy:", accuracy_score(y_test, y_pred))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

# Cross-validation to find the best hyper-parameter
depths = [1, 2, 3, 5, 10, None]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

best_depth = None
best_score = 0

print("\nCross-validation results:")
for d in depths:
    clf_cv = DecisionTreeClassifier(max_depth=d, random_state=42)
    scores = cross_val_score(clf_cv, X_train, y_train, cv=cv, scoring="accuracy")
    mean_score = scores.mean()
    print(f"max_depth = {d} -> mean CV accuracy = {mean_score:.4f}")
    
    if mean_score > best_score:
        best_score = mean_score
        best_depth = d

print("\nBest max_depth:", best_depth)
print("Best CV accuracy:", best_score)



Test accuracy: 0.5776666666666667
Confusion matrix:
[[617 224 159]
 [201 518 281]
 [154 248 598]]

Cross-validation results:
max_depth = 1 -> mean CV accuracy = 0.4267
max_depth = 2 -> mean CV accuracy = 0.4793
max_depth = 3 -> mean CV accuracy = 0.5412
max_depth = 5 -> mean CV accuracy = 0.5700
max_depth = 10 -> mean CV accuracy = 0.5963
max_depth = None -> mean CV accuracy = 0.5675

Best max_depth: 10
Best CV accuracy: 0.5962666666666667


## 3. Random Forests

[Random Forest](http://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html) classifiers use multiple decision trees trained on "weaker" datasets (less data and/or less features), averaging the results so as to reduce over-fitting.

* Use scikit-learn to **create a Random Forest classifier** on the CIFAR data. 
* Use cross-validation to find the best hyper-paramters for this method.

In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, confusion_matrix
from lab_tools import CIFAR10

if 'dataset' not in globals():
    dataset = CIFAR10('./CIFAR10')

X_train = dataset.train['hog']
y_train = dataset.train['labels']
X_test = dataset.test['hog']
y_test = dataset.test['labels']

# Simple Random Forest
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

# Test performance
y_pred = clf.predict(X_test)
print("Test accuracy:", accuracy_score(y_test, y_pred))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

# Cross-validation to find the best hyper-parameters
n_trees = [10, 50, 100]
depths = [None, 5, 10]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

best_score = 0
best_params = None

print("\nCross-validation results:")
for n in n_trees:
    for d in depths:
        clf_cv = RandomForestClassifier(
            n_estimators=n,
            max_depth=d,
            random_state=42
        )
        scores = cross_val_score(clf_cv, X_train, y_train, cv=cv, scoring="accuracy")
        mean_score = scores.mean()
        print(f"n_estimators = {n}, max_depth = {d} -> mean CV accuracy = {mean_score:.4f}")
        
        if mean_score > best_score:
            best_score = mean_score
            best_params = (n, d)

print("\nBest parameters:")
print("n_estimators =", best_params[0])
print("max_depth =", best_params[1])
print("Best CV accuracy:", best_score)


Test accuracy: 0.772
Confusion matrix:
[[797 152  51]
 [121 738 141]
 [ 59 160 781]]

Cross-validation results:
n_estimators = 10, max_depth = None -> mean CV accuracy = 0.6683
n_estimators = 10, max_depth = 5 -> mean CV accuracy = 0.6694
n_estimators = 10, max_depth = 10 -> mean CV accuracy = 0.6887
n_estimators = 50, max_depth = None -> mean CV accuracy = 0.7427
n_estimators = 50, max_depth = 5 -> mean CV accuracy = 0.7017
n_estimators = 50, max_depth = 10 -> mean CV accuracy = 0.7420
n_estimators = 100, max_depth = None -> mean CV accuracy = 0.7571
n_estimators = 100, max_depth = 5 -> mean CV accuracy = 0.7085
n_estimators = 100, max_depth = 10 -> mean CV accuracy = 0.7481

Best parameters:
n_estimators = 100
max_depth = None
Best CV accuracy: 0.7571333333333332
